# 05_screening
Scoping and pilot screening (Review Universe), session S3, after Search Strategy version 1 was certified by Uday (L000489). Records to screen: PubMed search 4 (L000459; records fetched L000491) and ClinicalTrials.gov search 5 (L000468). Duplicates removed by rule step9-dedup-1; order of screening by rule step9-order-2 (the assistant's suggestion). Screening decisions are Uday's only (k.screen); nothing here admits or excludes a record. Retrieval cells run only when RETRIEVE is True; the last cell recomputes every logged count from the saved raw files.

In [ ]:
import os, sys, json, csv
sys.path.insert(0, os.path.abspath('../env'))
import stir_l4_kit as k
k.use(os.path.abspath('..'))
import stir_l4_local as L
RETRIEVE = False

In [ ]:
# Retrieval: the PubMed records of search 4 (identifiers saved with L000459), saved in pages of 200 (L000491).
if RETRIEVE:
    k.pubmed_fetch(L.query_ids('L000459'), element='LU')

In [ ]:
# Records to screen, duplicates removed by PMID, then DOI, then normalized title and year (title-and-year matches kept
# apart for Uday to confirm); register entries listing a PubMed record of the pool as their own report are linked,
# not removed (rule step9-dedup-1, L000492). Grain: records.
pool = L.screening_pool('L000459', 'L000491', 'L000468')
rep = pool['report']
print({x: rep[x] for x in ('before', 'after', 'pubmed_records_saved', 'register_entries', 'removed_as_exact_duplicates')},
      rep['title_year_to_confirm'], len(rep['register_entries_linked']))

In [ ]:
# Order for screening, rule step9-order-1 (L000493; superseded before it was shown, note L000496) and
# rule step9-order-2 (L000494, used). Grain: PubMed records.
o1 = L.screening_order('L000491', 'L000459', added_lid='L000461')
o2 = L.screening_order('L000491', 'L000459', added_lid='L000461', out='data/derived/step9_order_rule2.csv', rule=L.RULE_9_ORDER_2)
print(o1['by_marks'], o2['by_marks'])

In [ ]:
# Records naming strength or physical performance but no fat, lean or fat-free mass, muscle itself or body water,
# and how many name heart failure (L000495). Grain: PubMed records.
fo = L.function_only('L000494', 'L000491')
print(fo['n'], fo['of'], fo['hf'], fo['top'], fo['n_top'])

In [ ]:
# Batch 1 (L000497): the first 10 records by rule step9-order-2, with the assistant's DRAFT suggestions (L000500);
# every quoted passage checked as written against the saved title and abstract.
batch = json.load(open(os.path.join(k._ROOT, 'data', 'derived', 'step9_batch1_suggestions.json'), encoding='utf-8'))
rows = L.batch_file(batch, 'L000491', 'data/derived/step9_pool.csv', out='data/derived/step9_batch1.csv')
print([(r['pmid'], r['passage_found']) for r in rows])

## After batch 1
Uday's decisions on batch 1 (answer L000505; screening decisions L000506–L000515, recorded with k.screen, never by code) and his confirmation that PMID 25166023 and 25268802 are one paper. The cells below recount from the saved files.

In [ ]:
# Title-and-year match confirmed by Uday (L000505): 25166023 kept, 25268802 (a correction notice) linked to it; records to
# screen before and after (L000517). Grain: records.
dd = L.dedup_confirmed(report_lid='L000492', answer_lid='L000505', keep={'PMID 25268802|PMID 25166023': 'PMID 25166023'},
                       efetch_lid='L000491')
print(dd['before'], dd['after'])

In [ ]:
# Screening decisions so far, and how many of the records added by the last change to the strings were screened (L000518).
ta = L.screening_tally(added_lid='L000461')
print(ta['screened'], ta['by'], ta['added_screened'], ta['added_total'])

In [ ]:
# Reports that may describe one trial (rule step9-links-1, L000519) and register entries linked to batches 1 and 2 (L000520).
b1 = ['29235586', '27550386', '28345814', '30675708', '31218819', '32568464', '33091173', '33186500', '33197559', '33351892']
b2 = [b['pmid'] for b in json.load(open(os.path.join(k._ROOT, 'data', 'derived', 'step9_batch2_suggestions.json'), encoding='utf-8'))]
sl = L.study_links(b1, 'L000491')
le = L.linked_entries_check(b1 + b2, 'L000468')
print(sl['pairs'])
print([(x['nct'], x['pmid'], x['n_listed']) for x in le['rows']])

In [ ]:
# Batch 2 (L000522): records 11-20 by rule step9-order-2, with the assistant's DRAFT suggestions (L000525).
batch2 = json.load(open(os.path.join(k._ROOT, 'data', 'derived', 'step9_batch2_suggestions.json'), encoding='utf-8'))
rows2 = L.batch_file(batch2, 'L000491', 'data/derived/step9_pool.csv', out='data/derived/step9_batch2.csv', start=11)
print([(r['n'], r['pmid'], r['passage_found']) for r in rows2])

## After batch 2
Uday's decisions on batch 2 (answer L000529; screening decisions L000530–L000539, recorded with k.screen, never by code): unsure 11, 13, 15, 17, 20; exclude 12, 14, 16, 18, 19. The cells below recount from the saved files and build batch 3, the last of the records with all three marks.

In [ ]:
# Screening decisions so far, and how many of the records added by the last change to the strings were screened (L000540).
ta2 = L.screening_tally(added_lid='L000461')
print(ta2['screened'], ta2['by'], ta2['added_screened'], ta2['added_total'])

In [ ]:
# Batch 3 (L000542): records 21-26 by rule step9-order-2, with the assistant's DRAFT suggestions (L000549).
batch3 = json.load(open(os.path.join(k._ROOT, 'data', 'derived', 'step9_batch3_suggestions.json'), encoding='utf-8'))
rows3 = L.batch_file(batch3, 'L000491', 'data/derived/step9_pool.csv', out='data/derived/step9_batch3.csv', start=21)
print([(r['n'], r['pmid'], r['passage_found'], r['linked_register_entries']) for r in rows3])

In [ ]:
# Reports that may describe one trial among the 26 records shown (rule step9-links-1, L000543) and register entries
# linked to batch 3 (L000544).
b3 = [b['pmid'] for b in batch3]
sl3 = L.study_links(b1 + b2 + b3, 'L000491', out='data/derived/step9_study_links_b3.csv')
le3 = L.linked_entries_check(b3, 'L000468', out='data/derived/step9_linked_entries_b3.csv')
print(sl3['pairs'])
print([(x['nct'], x['pmid'], x['n_listed']) for x in le3['rows']])

In [ ]:
# Retrieval: PMID 42286641, listed by register entry NCT05042973 as its own report (L000545).
if RETRIEVE:
    k.pubmed_fetch(['42286641'], element='SS')

In [ ]:
# The Empire Prevent register entries and the PubMed records each lists as its own reports (L000546); PMID 42286641
# was re-fetched above (L000545) but is already saved with L000491 (order 267; note L000548).
rr = L.register_entry_reports(['NCT05042973', 'NCT05084235'], 'L000468', pmids_of_interest=['38365073', '41507062', '42286641'])
print([(r['nct'], r['own_pmids'], r['order_of_listed'], r['not_listed']) for r in rr['rows']])

In [ ]:
# The saved pages of L000491 (11 files) checked in the order saved: per-page sha256 against the manifest and the
# combined sha256 against the log (L000555; note L000556: k.verify reads the pages in file-name order).
print(L.check_paged_hash('L000491'))

## After batch 3
Uday's decisions on batch 3 (answer L000557; screening decisions L000558–L000563, recorded with k.screen, never by code): include 26; unsure 22, 24, 25; exclude 21, 23. He kept the assistant's added rows: one study key per trial (EMPA-TROPISM; EMPERIAL; Empire Prevent Metabolic), the 2014 pair, register entries after the articles. The cells below recount from the saved files.

In [ ]:
# Study keys Uday accepted (L000564; rule step9-keys-1). Grain: records and study keys; a key may hold more than one trial.
groups = {'EMPA-TROPISM': ['PMID 30675708', 'PMID 33197559'],
          'EMPERIAL (two trials, reported together)': ['PMID 31218819', 'PMID 33351892'],
          'Empire Prevent Metabolic': ['PMID 38365073', 'PMID 41507062', 'PMID 42286641']}
sk = L.study_keys_confirmed(groups, answer_lid='L000557')
print(sk['records'], sk['keys'], sk['by_key'])

In [ ]:
# Screening decisions so far (L000565) and records left to screen (L000566). Grain: records.
ta3 = L.screening_tally(added_lid='L000461')
print(ta3['screened'], ta3['by'], ta3['added_screened'], ta3['added_total'])
rem = L.screening_remaining()
print(rem['total'], rem['screened'], rem['left'], rem['by_source'])

In [ ]:
# Atlas compartments named by the included and unsure records. L000567 used rule step7-rule-1, which misses the
# abbreviation TBW (correction L000570); L000569 uses rule step9-comp-1 and is the one reported.
cn1 = L.compartments_named('L000491')
cn2 = L.compartments_named('L000491', rule=L.RULE_9_COMP)
print(cn1['result']['include']['by_key'], cn2['result']['include']['by_key'], cn2['result']['unsure'])

## Commitment (Step 12, session S4)
Uday's instruction, kept as given (L000712): 'c. commit to the studies included so far'; commitment L000713. The cells below recount from the saved files; nothing here admits or excludes a record.

In [ ]:
# Final counts at the commitment (L000714; rule step12-counts-1). Grain: records and study keys.
cc = L.commitment_counts(commitment_lid='L000713')
print(cc['result'])

In [ ]:
# Page-order re-check of the 11-page response L000491 at the commitment (L000717; kit verify L000716).
print(L.check_pages('L000491', verify_lid='L000716'))

## Final plan (end of planning, session S4)
Uday's choice b, kept as given (L000722): the commitment stands and the certified search and methods stand as certified, the conflict recorded (L000723). The cell below assembles the final plan from the saved files only (L000724); it types no number.

In [ ]:
# Final plan (L000724; re-run after the question L000725, L000726): reports/2026-10-25_S4_final_plan.md
fp = L.final_plan_doc(counts_lid='L000714', commitment_lid='L000713', conflict_lids=['L000719', 'L000723'],
                      registration_lids=['L000708', 'L000709'], choice_lid='L000722',
                      out='reports/2026-10-25_S4_final_plan.md')
print(fp['path'])

In [ ]:
print(k.verify()['md'])